# Advanced: Multi-Strategy Data Cleaning with PAMOLA.CORE

**Goal**: Master all 3 constraint validation strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Range-based validation for numeric fields
- **Strategy 2**: Pattern-based validation for text/categorical fields
- **Strategy 3**: Comprehensive multi-field validation with null replacement
- Calculate advanced data quality metrics
- Analyze information loss from cleaning
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of data quality concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/cleaning/
        └── 02_clean_invalid_values_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.cleaning.clean_invalid_values import (
        CleanInvalidValuesOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 6 columns)
- Sample records (first 5 rows)
- Column statistics (types, ranges, unique counts)

**Note:** Generated data includes intentional invalid values (negative years, bad emails, out-of-range scores, invalid dates)

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'clean_invalid_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate data with various invalid values
    years_exp = np.random.randint(-10, 60, n_records)  # Some negative
    scores = np.random.uniform(-20, 120, n_records)    # Some out of range
    
    categories = ['A', 'B', 'C', 'D', 'E', 'Invalid', 'Unknown', 'BadValue']
    category_data = np.random.choice(categories, n_records)
    
    # Email with various invalid formats - FIX: Use np.random.choice for exact count
    valid_email_pool = [f'user{i}@example.com' for i in range(n_records)]
    invalid_email_pool = ['invalid-email', 'noemail', 'bad format', 'invalid@', '@nodomain.com']
    
    # Mix valid (70%) and invalid (30%) emails
    email_choices = np.random.choice(['valid', 'invalid'], n_records, p=[0.7, 0.3])
    email_data = [
        np.random.choice(valid_email_pool) if choice == 'valid' 
        else np.random.choice(invalid_email_pool)
        for choice in email_choices
    ]
    
    # Dates - Use np.random.choice for exact count
    valid_date_pool = pd.date_range('2020-01-01', '2024-12-31', periods=300).astype(str).tolist()
    invalid_date_pool = ['2030-01-01', '1990-05-20', '2025-13-45', 'invalid-date', '2024-99-99']
    
    # Mix valid (75%) and invalid (25%) dates
    date_choices = np.random.choice(['valid', 'invalid'], n_records, p=[0.75, 0.25])
    date_data = [
        np.random.choice(valid_date_pool) if choice == 'valid'
        else np.random.choice(invalid_date_pool)
        for choice in date_choices
    ]
    
    # Verify lengths before creating DataFrame
    print(f"🔍 Verifying array lengths:")
    print(f"  record_id: {n_records}")
    print(f"  years_exp: {len(years_exp)}")
    print(f"  scores: {len(scores)}")
    print(f"  category: {len(category_data)}")
    print(f"  email: {len(email_data)}")
    print(f"  date: {len(date_data)}")
    print()
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'years_of_experience': years_exp,
        'score': scores,
        'category': category_data,
        'email': email_data,
        'date': date_data
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:25s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:25s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 validation strategies!")

## Step 3: Prepare Whitelist/Blacklist Files

**How to use:**
- Run to check/create validation files
- Creates whitelist for valid categories
- Creates blacklist for invalid categories

**What you'll see:**
- File status (✓ found or 🔧 created)
- Whitelist: allowed categories (A, B, C, D, E)
- Blacklist: forbidden categories (Invalid, Unknown, BadValue)
- File locations in examples/data_examples/

**Note:** These files are used in Strategy 3 for categorical validation

In [ ]:
# Define file paths
examples_dir = project_root / 'examples'
whitelist_path = examples_dir / 'data_examples' / 'category_whitelist.txt'
blacklist_path = examples_dir / 'data_examples' / 'category_blacklist.txt'

# Check/create whitelist
if whitelist_path.exists():
    print(f"✓ Found existing whitelist: {whitelist_path}")
    with open(whitelist_path, 'r') as f:
        whitelist_items = f.read().splitlines()
    print(f"   Items: {', '.join(whitelist_items)}")
else:
    print(f"🔧 Creating whitelist file...")
    whitelist_items = ['A', 'B', 'C', 'D', 'E']
    try:
        whitelist_path.parent.mkdir(parents=True, exist_ok=True)
        with open(whitelist_path, 'w') as f:
            f.write('\n'.join(whitelist_items))
        print(f"✓ Whitelist created: {whitelist_path}")
        print(f"   Allowed categories: {', '.join(whitelist_items)}")
    except PermissionError:
        print(f"⚠️  Cannot save to {whitelist_path} (file may be open)")

# Check/create blacklist
if blacklist_path.exists():
    print(f"\n✓ Found existing blacklist: {blacklist_path}")
    with open(blacklist_path, 'r') as f:
        blacklist_items = f.read().splitlines()
    print(f"   Items: {', '.join(blacklist_items)}")
else:
    print(f"\n🔧 Creating blacklist file...")
    blacklist_items = ['Invalid', 'Unknown', 'BadValue']
    try:
        blacklist_path.parent.mkdir(parents=True, exist_ok=True)
        with open(blacklist_path, 'w') as f:
            f.write('\n'.join(blacklist_items))
        print(f"✓ Blacklist created: {blacklist_path}")
        print(f"   Forbidden categories: {', '.join(blacklist_items)}")
    except PermissionError:
        print(f"⚠️  Cannot save to {blacklist_path} (file may be open)")

print("\n💡 These files will be used in Strategy 3!")
print("=" * 80)

## Step 4: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `strategy1_fields`: Numeric fields for range validation
   - `strategy2_fields`: Text/categorical fields for pattern validation
   - `strategy3_fields`: All fields for comprehensive validation
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_fields": ["years_of_experience", "score"],           # Range-based
    "strategy2_fields": ["email", "category"],                      # Pattern-based
    "strategy3_fields": ["date", "category", "years_of_experience"] # Comprehensive
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
all_fields = set()
for strategy, field_list in FIELD_CONFIG.items():
    for field_name in field_list:
        all_fields.add(field_name)
        if field_name not in df.columns:
            raise ValueError(
                f"❌ Field '{field_name}' for {strategy} not found!\n"
                f"Available columns: {', '.join(df.columns)}\n"
                f"Please update FIELD_CONFIG"
            )

for field_name in sorted(all_fields):
    print(f"  ✓ {field_name:25s}: {df[field_name].nunique()} unique values")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy_cleaning",
    description="Multi-strategy data cleaning with constraint validation",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Range-Based Validation

**How to use:**
- Validates numeric fields against min/max ranges
- Run to execute range-based strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Invalid value counts (e.g., 150 out-of-range values)

**Key parameters:**
- `constraint_type='valid_range'`: Validates min and max bounds
- `years_of_experience`: 0-50 range (removes negatives)
- `score`: 0-100 range (removes out-of-range scores)
- `mode='ENRICH'`: Keeps original + adds cleaned field

**Note:** Invalid values replaced with null. Best for numeric data quality

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: RANGE-BASED VALIDATION")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Range-based",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = CleanInvalidValuesOperation(
    # Field constraints for numeric validation
    field_constraints={
        "years_of_experience": {
            "data_type": "numeric",
            "constraint_type": "valid_range",
            "min_value": 1,
            "max_value": 50
        },
        "score": {
            "data_type": "numeric",
            "constraint_type": "valid_range",
            "min_value": 1,
            "max_value": 100
        }
    },
    
    # Mode configuration
    mode='ENRICH',
    column_prefix='clean_',
    
    # Null replacement
    null_replacement=None,  # Keep as null
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_range',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_range' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    
    # Count invalid values
    for field in FIELD_CONFIG["strategy1_fields"]:
        clean_field = f"clean_{field}"
        if clean_field in result_df_s1.columns:
            invalid_count = result_df_s1[clean_field].isna().sum() - df[field].isna().sum()
            print(f"\n📊 {field}: {invalid_count} invalid values removed")

## STRATEGY 2: Pattern-Based Validation

**How to use:**
- Validates text/categorical fields using patterns
- Run to execute pattern-based strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Pattern match statistics (e.g., 200 invalid emails)

**Key parameters:**
- `constraint_type='valid_pattern'`: Regex-based validation
- `email`: Email format validation (user@domain.ext)
- `constraint_type='allowed_values'`: Whitelist validation
- `category`: Only A, B, C, D, E allowed

**Note:** Best for text validation and categorical constraints

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: PATTERN-BASED VALIDATION")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Pattern-based",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = CleanInvalidValuesOperation(
    # Field constraints
    field_constraints={
        "email": {
            "data_type": "text",
            "constraint_type": "valid_pattern",
            "valid_pattern": r'^[\w\.-]+@[\w\.-]+\.\w+$'
        },
        "category": {
            "data_type": "categorical",
            "constraint_type": "allowed_values",
            "allowed_values": ['A', 'B', 'C', 'D', 'E']
        }
    },
    
    mode='ENRICH',
    column_prefix='clean_',
    null_replacement=None,
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_pattern',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_pattern' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    
    for field in FIELD_CONFIG["strategy2_fields"]:
        clean_field = f"clean_{field}"
        if clean_field in result_df_s2.columns:
            invalid_count = result_df_s2[clean_field].isna().sum() - df[field].isna().sum()
            print(f"\n📊 {field}: {invalid_count} invalid values removed")

## STRATEGY 3: Comprehensive Multi-Field Validation

**How to use:**
- Combines multiple validation strategies
- Uses whitelist/blacklist files
- Applies null replacement strategies
- Run to execute comprehensive validation

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Multi-field validation statistics

**Key parameters:**
- `date_range`: 2020-01-01 to 2024-12-31
- `whitelist_file`: External validation file
- `null_replacement`: Mean for numeric, mode for categorical

**Note:** Most comprehensive strategy - production ready

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: COMPREHENSIVE MULTI-FIELD VALIDATION")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Comprehensive",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = CleanInvalidValuesOperation(
    # Field constraints
    field_constraints={
        "date": {
            "data_type": "date",
            "constraint_type": "date_range",
            "min_date": "2020-01-01",
            "max_date": "2024-12-31"
        },
        "years_of_experience": {
            "data_type": "numeric",
            "constraint_type": "valid_range",
            "min_value": 1,
            "max_value": 50
        }
    },
    
    # Whitelist/blacklist validation
    whitelist_path={"category": whitelist_path},
    blacklist_path=None,
    
    # Null replacement strategies
    null_replacement={
        "years_of_experience": "mean",
        "category": "mode"
    },
    
    mode='ENRICH',
    column_prefix='clean_',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_comprehensive',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_comprehensive' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    
    for field in FIELD_CONFIG["strategy3_fields"]:
        clean_field = f"clean_{field}"
        if clean_field in result_df_s3.columns:
            original_nulls = df[field].isna().sum()
            cleaned_nulls = result_df_s3[clean_field].isna().sum()
            print(f"\n📊 {field}:")
            print(f"   Invalid removed: {cleaned_nulls - original_nulls}")
            print(f"   Nulls replaced: {max(0, original_nulls - cleaned_nulls)}")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Invalid value removal counts
- Strategy effectiveness comparison

**Note:** Fastest strategy isn't always best - consider data quality needs

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1: {elapsed_s1:6.2f}s")
print(f"  Strategy 2: {elapsed_s2:6.2f}s")
print(f"  Strategy 3: {elapsed_s3:6.2f}s")
print(f"  Total:      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print("\n📈 Data Quality Impact:")
if 'result_df_s1' in locals():
    s1_cleaned = sum([
        result_df_s1[f'clean_{f}'].isna().sum() - df[f].isna().sum()
        for f in FIELD_CONFIG["strategy1_fields"]
        if f'clean_{f}' in result_df_s1.columns
    ])
    print(f"  Strategy 1: {s1_cleaned} invalid values removed")

if 'result_df_s2' in locals():
    s2_cleaned = sum([
        result_df_s2[f'clean_{f}'].isna().sum() - df[f].isna().sum()
        for f in FIELD_CONFIG["strategy2_fields"]
        if f'clean_{f}' in result_df_s2.columns
    ])
    print(f"  Strategy 2: {s2_cleaned} invalid values removed")

if 'result_df_s3' in locals():
    s3_cleaned = sum([
        max(0, result_df_s3[f'clean_{f}'].isna().sum() - df[f].isna().sum())
        for f in FIELD_CONFIG["strategy3_fields"]
        if f'clean_{f}' in result_df_s3.columns
    ])
    print(f"  Strategy 3: {s3_cleaned} invalid values (with replacement)")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Invalid counts, constraint violations, performance
2. **Visualizations**: Charts displayed inline (latest batch only)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_range', 'Strategy 1: Range-Based'),
    ('strategy2_pattern', 'Strategy 2: Pattern-Based'),
    ('strategy3_comprehensive', 'Strategy 3: Comprehensive')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display key metrics
                if 'invalid_values' in metrics:
                    print(f"\n   Invalid Values:")
                    for field, stats in metrics['invalid_values'].items():
                        print(f"      {field}: {stats.get('count', 0)} ({stats.get('percent', 0):.1%})")
                    
                if 'execution_time_seconds' in metrics:
                    print(f"\n   Execution Time: {metrics['execution_time_seconds']:.4f}s")
                    
                if 'records_processed' in metrics:
                    print(f"   Records Processed: {metrics['records_processed']:,}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Calculate Data Quality Metrics

**How to use:**
- Calculate quality improvements from cleaning
- Run AFTER all 3 strategies complete

**What you'll see:**
- Original data: null counts, invalid value estimates
- After cleaning: reduced nulls, improved validity
- Quality improvement ratios (e.g., 2.5x fewer invalid values)

**Quality targets:**
- <1% invalid values: Excellent
- <5% invalid values: Good
- <10% invalid values: Acceptable

**Note:** Strategy 3 shows best results due to null replacement

In [ ]:
print("\n" + "=" * 80)
print("📊 DATA QUALITY METRICS")
print("=" * 80 + "\n")

def calculate_quality_score(df, fields):
    """Calculate data quality score based on null/invalid values"""
    total_values = len(df) * len(fields)
    null_count = sum(df[f].isna().sum() for f in fields if f in df.columns)
    quality_score = (total_values - null_count) / total_values
    return {
        'total_values': total_values,
        'null_count': null_count,
        'valid_count': total_values - null_count,
        'quality_score': quality_score
    }

# Original data quality
all_fields = list(set([f for fields in FIELD_CONFIG.values() for f in fields]))
orig_quality = calculate_quality_score(df, all_fields)

print(f"📊 ORIGINAL DATA:")
print(f"   Total values: {orig_quality['total_values']:,}")
print(f"   Null count: {orig_quality['null_count']:,}")
print(f"   Quality score: {orig_quality['quality_score']:.2%}")

# Strategy 1 quality
if 'result_df_s1' in locals():
    clean_fields_s1 = [f'clean_{f}' for f in FIELD_CONFIG['strategy1_fields']]
    s1_quality = calculate_quality_score(result_df_s1, clean_fields_s1)
    
    print(f"\n✨ STRATEGY 1 (Range-Based):")
    print(f"   Null count: {s1_quality['null_count']:,}")
    print(f"   Quality score: {s1_quality['quality_score']:.2%}")
    
    if orig_quality['quality_score'] > 0:
        improvement = s1_quality['quality_score'] / orig_quality['quality_score']
        print(f"   Improvement: {improvement:.2f}x")

# Strategy 3 quality (with replacement)
if 'result_df_s3' in locals():
    clean_fields_s3 = [f'clean_{f}' for f in FIELD_CONFIG['strategy3_fields']]
    s3_quality = calculate_quality_score(result_df_s3, clean_fields_s3)
    
    print(f"\n✨ STRATEGY 3 (Comprehensive):")
    print(f"   Null count: {s3_quality['null_count']:,}")
    print(f"   Quality score: {s3_quality['quality_score']:.2%}")
    
    if orig_quality['quality_score'] > 0:
        improvement = s3_quality['quality_score'] / orig_quality['quality_score']
        print(f"   Improvement: {improvement:.2f}x")

print(f"\n💡 Strategy 3 shows best quality due to null replacement!")

## Step 8: Export Final Data

**How to use:**
- Export final cleaned datasets from all strategies
- Run AFTER all 3 strategies complete

**What you'll see (per strategy):**
- Available columns list
- Export confirmation (file path, column/row count)
- Preview of first 5 rows
- Quality statistics (null counts before/after)

**Output structure:**
```
advanced_output/
├── strategy1_range/range_cleaned_data.csv
├── strategy2_pattern/pattern_cleaned_data.csv
└── strategy3_comprehensive/comprehensive_cleaned_data.csv
```

**Note:** Files include both original and cleaned columns for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Range-Based Validation
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: RANGE-BASED VALIDATION")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_range'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    # Select columns to export
    export_cols_s1 = ['record_id'] + FIELD_CONFIG['strategy1_fields'] + \
                     [f'clean_{f}' for f in FIELD_CONFIG['strategy1_fields']]
    existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
    
    final_df_s1 = result_df_s1[existing_cols_s1].copy()
    
    # Save
    output_path_s1 = s1_dir / 'range_cleaned_data.csv'
    try:
        final_df_s1.to_csv(output_path_s1, index=False)
        print(f"\n✅ Saved: {output_path_s1}")
        print(f"   Rows: {len(final_df_s1):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    # Preview
    print(f"\n📊 Preview:")
    print(final_df_s1.head())
    
    # Statistics
    print(f"\n📈 Quality Statistics:")
    for field in FIELD_CONFIG['strategy1_fields']:
        clean_field = f'clean_{field}'
        if clean_field in final_df_s1.columns:
            orig_nulls = final_df_s1[field].isna().sum()
            clean_nulls = final_df_s1[clean_field].isna().sum()
            removed = clean_nulls - orig_nulls
            print(f"   {field}: {removed} invalid values removed")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Pattern-Based Validation
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: PATTERN-BASED VALIDATION")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_pattern'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    export_cols_s2 = ['record_id'] + FIELD_CONFIG['strategy2_fields'] + \
                     [f'clean_{f}' for f in FIELD_CONFIG['strategy2_fields']]
    existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
    
    final_df_s2 = result_df_s2[existing_cols_s2].copy()
    
    output_path_s2 = s2_dir / 'pattern_cleaned_data.csv'
    try:
        final_df_s2.to_csv(output_path_s2, index=False)
        print(f"\n✅ Saved: {output_path_s2}")
        print(f"   Rows: {len(final_df_s2):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(final_df_s2.head())
    
    print(f"\n📈 Quality Statistics:")
    for field in FIELD_CONFIG['strategy2_fields']:
        clean_field = f'clean_{field}'
        if clean_field in final_df_s2.columns:
            orig_nulls = final_df_s2[field].isna().sum()
            clean_nulls = final_df_s2[clean_field].isna().sum()
            removed = clean_nulls - orig_nulls
            print(f"   {field}: {removed} invalid values removed")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Comprehensive Multi-Field Validation
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: COMPREHENSIVE MULTI-FIELD VALIDATION")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_comprehensive'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    export_cols_s3 = ['record_id'] + FIELD_CONFIG['strategy3_fields'] + \
                     [f'clean_{f}' for f in FIELD_CONFIG['strategy3_fields']]
    existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
    
    final_df_s3 = result_df_s3[existing_cols_s3].copy()
    
    output_path_s3 = s3_dir / 'comprehensive_cleaned_data.csv'
    try:
        final_df_s3.to_csv(output_path_s3, index=False)
        print(f"\n✅ Saved: {output_path_s3}")
        print(f"   Rows: {len(final_df_s3):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(final_df_s3.head())
    
    print(f"\n📈 Quality Statistics:")
    for field in FIELD_CONFIG['strategy3_fields']:
        clean_field = f'clean_{field}'
        if clean_field in final_df_s3.columns:
            orig_nulls = final_df_s3[field].isna().sum()
            clean_nulls = final_df_s3[clean_field].isna().sum()
            if clean_nulls > orig_nulls:
                print(f"   {field}: {clean_nulls - orig_nulls} invalid removed")
            else:
                print(f"   {field}: {orig_nulls - clean_nulls} nulls replaced")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 All files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 validation strategies implemented and compared
- ✅ Data quality metrics calculated
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Range-based validation best for numeric data quality
- Pattern-based validation essential for text/categorical fields
- Comprehensive strategy with null replacement maximizes data completeness

**Strategy recommendations:**
- **Use Strategy 1** when: Numeric data with clear boundaries (age, scores, counts)
- **Use Strategy 2** when: Text patterns (emails, IDs) or categorical constraints
- **Use Strategy 3** when: Production systems requiring high data completeness

**Next steps:**
- Test with your own datasets
- Tune constraint parameters for optimal results
- Deploy to production data pipelines

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)